# 40 · Which PT zonation programs are conserved, reversed or species-specific?

Notebook 39 showed two things about human and mouse PT:
- their zonation programs are largely species-specific (low cross-species correlation of segment
  gradients);
- overall human amplitude is moderately lower.

This notebook classifies each gene and pathway, and builds the main figure for paper section R3.
Segment labels define every contrast, so nothing depends on the reconstructed coordinate.

## Pre-registered protocol (fixed before any result in this notebook was computed)

**Contrasts** (pseudobulk log2 CPM ratios per specimen, as in notebook 39):
- **S2 − S1**, cortical in both species (primary);
- **S3c − early**, with mouse S3 restricted to cortical-like structures (region-matched; primary);
- **S3 − early**, all S3 (reported, region-mismatched).

**Per-species calls, scaled to each species' noise.**
- Within each species, take the moderated one-sample t of the specimen-mean contrast. Variances
  are shrunk within ten expression bins of that species, as in limma's squeezeVar. BH runs over
  genes.
- A gene is **zonated** when q ≤ 0.05 and |contrast| ≥ 0.5 log2, **and** the same-species snRNA
  reference confirms the direction: one-sided moderated donor-level p ≤ 0.05. The references are
  Census mouse snRNA (12 male donors) and Lake/KPMP healthy human cortex (7 donors).
- A gene is **flat** when its 95% interval lies inside ±0.3 log2.
- Anything else is **indeterminate**.

**Classes.**

| Class | Mouse | Human |
|---|---|---|
| conserved | zonated | zonated, same sign |
| reversal | zonated | zonated, opposite sign |
| mouse-only | zonated | flat |
| human-only | flat | zonated |
| neither | flat or not zonated | flat or not zonated, at least one flat |
| indeterminate | the rest | the rest |

A noisier species therefore cannot inflate the other species' "only" class: "only" needs the
other species to be confidently flat. The **gene class** combines S2 − S1 and S3c − early with the
precedence reversal > conserved > mouse-only > human-only.

**Gene universe and separate reporting.**
- Classified genes have ≥ 20 summed counts in every specimen × segment pseudobulk of both species.
- Genes expressed in one species only (≥ 20 in that species, < 20 in every pseudobulk of the
  other) are reported separately, with their zonation call in the expressing species.
- Genes with unequal probe counts (workstream 3's table) are classified but reported separately.
  Headline counts use probe-balanced genes.

**Alternatives**, with class sizes reported under each:
- A1: no reference confirmation;
- A2: no effect floor (significance only);
- A3: effect floor scaled by each species' noise-corrected amplitude (human floor = 0.5 × the
  human ÷ mouse amplitude ratio from notebook 39's method);
- A4: flat margin 0.5;
- A5: mouse confirmation by male microdissection, same sign with |gradient| ≥ 0.5.

**Added in version 2 (after review):** two sex sensitivities. Our mice are male, so male
confirmation stays primary.
- A6: mouse calls confirmed by the 12 female Census snRNA donors.
- A7: mouse calls confirmed by female microdissection (GSE212213 means; same sign with
  |gradient| ≥ 0.5).

For each, we report how many primary mouse-only, conserved and reversal genes keep their class.

**Pathway enrichment per class.**
- Each class's membership indicator is tested with the competitive rank-AUC against random gene
  sets matched on mean expression, detection and minimum species detection (9,999 draws).
- The variance is inflated by a CAMERA factor from coordinate-free residual correlations
  (structure expression after removing specimen × segment means).
- BH runs within each class. Pathways have 10–300 classified members (Reactome 2022, Hallmark,
  KEGG 2019 Mouse).
- **Not expression-driven** means the pathway stays q ≤ 0.10 with 5-bin matching **and** among
  genes above the lowest expression tertile.
- Enriched pathways are collapsed into gene-overlap programs, using notebook 37's rule with the
  class genes as contributing members.

**Mapping.** The 36 robust pathways (notebook 37) are described by their members' class
composition; lead genes are tabulated by class.

**Representative genes** (for the figure, chosen by rule): probe-balanced and above-median
expression. The ranking is the largest min(|t_mouse|, |t_human|) for conserved and reversal, and
the largest |t| in the zonated species for the species-only classes.

## 1 · Folders and settings

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
upstream13 = results_root / 'minimal_pt_scfates'
upstream37 = results_root / 'pt_pathway_final' / 'scfates'
external = data_root / 'external'
output = results_root / 'pt_zonation_classes'
figures, tables = output / 'figures', output / 'tables'
for folder in (figures, tables):
    folder.mkdir(parents=True, exist_ok=True)
lake_file = external / 'kpmp_lake2023' / 'lake2023_snRNA_v1.5.h5ad'
probe_file = results_root / 'pt_literature_deep' / 'probe_counts.csv'
required = [upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv',
            results_root / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad',
            upstream37 / 'tables' / 'table_S2_pathways.csv',
            external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz',
            external / 'segment_microdissection_sex' / 'GSE212213_TPM_Male_Female.txt.gz',
            external / 'census_pt_segments' / 'mouse_pt_segment_pseudobulk_counts.csv.gz',
            external / 'census_pt_segments' / 'human_pt_segment_pseudobulk_counts.csv.gz', lake_file, probe_file]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '40.zonation_classes.2'
SEED, N_NULL = 40, 9999
MIN_COUNTS, MIN_NUCLEI, REFERENCE_LEVEL = 20, 50, 3.
ALPHA, EFFECT, MARGIN, CONFIRM_ALPHA, ENRICH_Q = .05, .5, .3, .05, .10
print('Results folder:', output)

## 2 · Our structures, counts and segment pseudobulks (as notebook 39)

- **Cross-species data.** All of notebook 13's PT structures with their reviewed S1/S2/S3 labels.
  No coordinate filter is applied.
- **Gene space.** Accepted orthologs measured in all four specimens (mouse symbols).
- **Library size.** The total over that gene space.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
from pseudospace.pathway_inputs import rebuild_pt_expression
from pseudospace.stage_cache import cached_payload, cached_frame, digest
from pseudospace import zonation_amplitude as amplitude_module
from pseudospace.zonation_amplitude import (group_sums, log_ratio, ols_slope, deming_slope, amplitude_ratio,
                                            bootstrap_ratio, flattening_class)

cache = output / 'stage_cache'
from pseudospace import zonation_classes as _classes_module
code = digest([NOTEBOOK_LOGIC_VERSION, Path(amplitude_module.__file__), Path(_classes_module.__file__)])
saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r')
pt_obs = saved_pt.obs.copy()
saved_pt.file.close()
pt_obs = pt_obs[pt_obs.broad_tubule_marker_call.astype(str).eq('PT')].copy()
expected = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
names = expected['mouse'] + expected['human']
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
source_paths = {name: data_root / 'tubule_by_gene' / f'{name}_tubule_by_gene_caleb.h5ad' for name in names}
adata = rebuild_pt_expression(pt_obs, source_paths, orthologs, target_sum=1e4)
measured = adata.var.measured_in_both_inputs.to_numpy()
gene_space = adata.var_names[measured]
counts = sparse.csr_matrix(adata.layers['counts'], dtype=float)[:, measured]
lognorm = sparse.csr_matrix(adata.layers['lognorm'], dtype=float)[:, measured]
specimen = adata.obs['sample'].astype(str).to_numpy()
segment = adata.obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
library = np.asarray(counts.sum(axis=1)).ravel()
display(pd.crosstab(specimen, segment))
structure_depth = pd.DataFrame({'specimen': specimen, 'library': library}).groupby('specimen').library.describe()[['count', '50%', 'mean']]
display(structure_depth.rename(columns={'50%': 'median counts per structure'}))

### Cortical-like mouse S3

Depth is the mean distance to the three nearest reviewed glomeruli of the same specimen, with
centroids verified against the fine GeoJSON (notebook 33's rule).

In [ ]:
from scipy.spatial import cKDTree
from pseudospace.repeated_inputs import _read_and_validate_segmentation


def mouse_depth():
    pass1 = ad.read_h5ad(results_root / 'human_vs_healthy_mouse' / 'cross_species_harmony_pass1.h5ad', backed='r')
    anchors = pass1.obs[['sample', 'coarse_class', 'feature_index', 'x_centroid', 'y_centroid']].copy()
    pass1.file.close()
    values = np.full(len(specimen), np.nan)
    for name in expected['mouse']:
        native = ad.read_h5ad(source_paths[name], backed='r')
        source_obs = native.obs[['feature_index', 'x_centroid', 'y_centroid']].copy()
        native.file.close()
        mask = specimen == name
        structures = source_obs.loc[adata.obs_names[mask].str[len(name) + 1:]]
        _read_and_validate_segmentation(data_root, name, structures)
        glomeruli = anchors[anchors['sample'].astype(str).eq(name) & anchors.coarse_class.astype(str).eq('Glomerulus')]
        _read_and_validate_segmentation(data_root, name, glomeruli)
        distance, _ = cKDTree(glomeruli[['x_centroid', 'y_centroid']].to_numpy(float)).query(
            structures[['x_centroid', 'y_centroid']].to_numpy(float), k=3)
        values[mask] = distance.mean(axis=1)
    return {'depth': values}


depth = cached_payload('mouse_depth', mouse_depth, root=cache, inputs={'obs': list(adata.obs_names)}, code=code)['depth']
cortical_s3 = np.zeros(len(specimen), bool)
for name in expected['mouse']:
    limit = np.quantile(depth[(specimen == name) & (segment == 'S1')], .95)
    cortical_s3 |= (specimen == name) & (segment == 'S3') & (depth <= limit)
segment_region = np.where((segment == 'S3') & np.isin(specimen, expected['mouse']) & ~cortical_s3, 'S3deep', segment)
print('Cortical-like mouse S3 structures:', {n: int((cortical_s3 & (specimen == n)).sum()) for n in expected['mouse']})

In [ ]:
def specimen_pseudobulk(count_matrix, specimen_labels, segment_labels, genes):
    """Summed counts per specimen and segment (S1, S2, S3, cortical-S3 and early = S1 + S2)."""
    names_, sums = group_sums(count_matrix, np.char.add(np.char.add(specimen_labels.astype(str), '|'), segment_labels.astype(str)))
    frame = pd.DataFrame(sums, index=names_, columns=genes)
    out = {}
    for name in np.unique(specimen_labels):
        rows = lambda labels: [f'{name}|{s}' for s in labels if f'{name}|{s}' in frame.index]
        out[(name, 'S1')] = frame.loc[rows(['S1'])].sum()
        out[(name, 'S2')] = frame.loc[rows(['S2'])].sum()
        out[(name, 'S3')] = frame.loc[rows(['S3', 'S3deep'])].sum()
        out[(name, 'S3c')] = frame.loc[rows(['S3'])].sum()
        out[(name, 'early')] = out[(name, 'S1')] + out[(name, 'S2')]
    return pd.DataFrame(out)


ours_counts = specimen_pseudobulk(counts, specimen, segment_region, gene_space)
ours_library = pd.Series({key: float(library[(specimen == key[0]) & np.isin(
    segment_region, {'S1': ['S1'], 'S2': ['S2'], 'S3': ['S3', 'S3deep'], 'S3c': ['S3'], 'early': ['S1', 'S2']}[key[1]])].sum())
    for key in ours_counts.columns})

## 3 · Our gradients

- **Pseudobulk log2 ratio (primary):** log2 CPM(late) − log2 CPM(early) from summed counts per
  specimen.
- **log1p mean (sensitivity):** mean log1p(CP10k) per segment, the contrast used in notebooks 34
  and 37.
- **Contrasts:**
  - `S2-S1`;
  - `S3c-early`, with mouse S3 restricted to cortical-like structures and human S3 complete (both
    human sections are cortex);
  - `S3-early`, all S3.

In [ ]:
CONTRASTS = {'S2-S1': ('S2', 'S1'), 'S3c-early': ('S3c', 'early'), 'S3-early': ('S3', 'early')}


def gradients(pb, pb_library, specimens):
    out = {}
    for contrast, (late, early) in CONTRASTS.items():
        for name in specimens:
            out[(contrast, name)] = log_ratio(pb[(name, late)], pb_library[(name, late)], pb[(name, early)], pb_library[(name, early)])
    return pd.DataFrame(out, index=pb.index)


ours = gradients(ours_counts, ours_library, names)
log1p_means = {}
for name in names:
    for label, parts in {'S1': ['S1'], 'S2': ['S2'], 'S3': ['S3', 'S3deep'], 'S3c': ['S3'], 'early': ['S1', 'S2']}.items():
        rows = (specimen == name) & np.isin(segment_region, parts)
        log1p_means[(name, label)] = np.asarray(lognorm[rows].mean(axis=0)).ravel()
log1p_means = pd.DataFrame(log1p_means, index=gene_space)
ours_log1p = pd.DataFrame({(contrast, name): log1p_means[(name, late)] - log1p_means[(name, early)]
                           for contrast, (late, early) in CONTRASTS.items() for name in names})


def count_filter(pb, specimens, parts):
    return pb[[(n, p) for n in specimens for p in parts]].min(axis=1).ge(MIN_COUNTS)


our_genes = {contrast: gene_space[count_filter(ours_counts, names, list(parts)).to_numpy()]
             for contrast, parts in CONTRASTS.items()}
def species_mean(frame, contrast, specimens):
    return frame[[(contrast, n) for n in specimens]].mean(axis=1)


print({c: len(g) for c, g in our_genes.items()}, 'genes pass the count filter (cross-species)')

## 4 · Independent references

Each reference gives a gene-level gradient for each contrast (log2 units), plus a donor-resampling
function for the bootstrap.
- **Microdissection male** (GSE150338): mean log2(TPM + 1) per mouse, then over mice. Segments
  come from different mice, so the bootstrap resamples mice within each segment.
- **Microdissection by sex** (GSE212213): segment means only, so the bootstrap resamples genes only.
- **Mouse snRNA** (Census): log2(CPM + 1) per donor; early = pooled S1 + S2 counts; donors with
  ≥ 50 nuclei in every segment.
- **Lake / KPMP human cortex** (CELLxGENE v1.5): PT-S1/S2/S3 nuclei of cortex samples, summed per
  donor and segment, with genes mapped by Ensembl id. Healthy donors are the human reference. All
  donors, healthy plus AKI and CKD, enter the tissue-state analysis.

For microdissection, "S3c" is the dissected S3, which lies in the outer stripe, so it equals
"S3".

In [ ]:
import re


def microdissection_male():
    table = pd.read_csv(external / 'segment_microdissection' / 'GSE150338_Mouse_TPM_Replicates.txt.gz', sep='\t')
    table = table.drop_duplicates('Gene_symbol').set_index('Gene_symbol')
    per_mouse = {}
    for column in table.columns:
        match = re.fullmatch(r'(m\d+)_PTS([123])_[a-z]', column)
        if match:
            per_mouse.setdefault((f'S{match.group(2)}', match.group(1)), []).append(column)
    values = {key: np.log2(table[cols].apply(pd.to_numeric, errors='coerce').clip(lower=0) + 1).mean(axis=1)
              for key, cols in per_mouse.items()}
    return pd.DataFrame(values).reindex(gene_space)


micro_male = microdissection_male()
micro_sex_table = pd.read_csv(external / 'segment_microdissection_sex' / 'GSE212213_TPM_Male_Female.txt.gz', sep='\t')
micro_sex_table = micro_sex_table.drop_duplicates('Gene_symbol').set_index('Gene_symbol')
micro_sex = {sex: pd.DataFrame({f'S{k}': np.log2(pd.to_numeric(micro_sex_table[f'{prefix}PTS{k}'], errors='coerce').clip(lower=0) + 1)
                                for k in (1, 2, 3)}).reindex(gene_space)
             for sex, prefix in (('male', ''), ('female', 'f'))}


def census_segments(label, segment_map):
    table = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    meta = meta[~meta.development_stage.str.contains('embryo|E16|Theiler', case=False)].copy()
    meta['segment'] = meta.cell_type.map(segment_map)
    meta = meta[meta.segment.notna() & meta.n_cells.ge(MIN_NUCLEI)]
    needed = set(segment_map.values())
    complete = meta.groupby('donor_id').segment.apply(lambda s: needed.issubset(set(s)))
    meta = meta[meta.donor_id.isin(complete[complete].index)]
    counts_out, totals, info = {}, {}, {}
    for (donor, sex), block in meta.groupby(['donor_id', 'sex']):
        for seg_name in needed:
            index = block.index[block.segment.eq(seg_name)]
            counts_out[(donor, seg_name)] = table.loc[index].sum()
            totals[(donor, seg_name)] = meta.loc[index, 'total_counts_all_genes'].sum()
        info[donor] = sex
    return pd.DataFrame(counts_out).reindex(gene_space).fillna(0), pd.Series(totals), pd.Series(info)


sn_counts, sn_totals, sn_sex = census_segments('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                                                          'epithelial cell of proximal tubule segment 2': 'S2',
                                                          'epithelial cell of proximal tubule segment 3': 'S3'})
census_h_counts, census_h_totals, census_h_sex = census_segments('human', {
    'kidney proximal convoluted tubule epithelial cell': 'early', 'epithelial cell of proximal tubule segment 3': 'S3'})


def lake_pseudobulk():
    """Lake/KPMP cortex PT nuclei summed per donor x disease x segment, plus PT-state counts per donor."""
    import h5py
    from pseudospace.replication_rules import csr_group_sums

    def column(group, name):
        item = group[name]
        if isinstance(item, h5py.Group):
            categories = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in item['categories'][:]])
            return pd.Categorical.from_codes(item['codes'][:], categories).astype(str)
        values = item[:]
        return values.astype(str) if values.dtype.kind in 'SO' else values

    with h5py.File(lake_file, 'r') as handle:
        obs = pd.DataFrame({c: column(handle['obs'], c) for c in ['subclass.l2', 'disease', 'region', 'tissue', 'sex', 'donor_id']})
        var_ids = handle['raw/var/_index'][:].astype(str)
        var_names = np.asarray(column(handle['raw/var'], 'feature_name'))
        cortex = obs.region.eq('C') & obs.tissue.eq('cortex of kidney')
        keep = cortex & obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3'])
        label = (obs.donor_id + '|' + obs.disease + '|' + obs.sex + '|' + obs['subclass.l2'].str.replace('PT-', '', regex=False))[keep]
        groups, codes = np.unique(label.to_numpy(), return_inverse=True)
        raw = handle['raw/X']
        sums, totals, nuclei = csr_group_sums(raw['data'], raw['indices'], raw['indptr'], np.flatnonzero(keep.to_numpy()),
                                              codes, len(groups), len(var_ids), block_rows=5000)
        states = obs[cortex & obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3', 'aPT', 'dPT', 'cycPT'])]
        state_counts = states.groupby(['donor_id', 'subclass.l2']).size().unstack(fill_value=0)
    human_source = ad.read_h5ad(source_paths[expected['human'][0]], backed='r')
    human_var = human_source.var[['gene_ids']].copy()
    human_source.file.close()
    by_id = dict(zip(orthologs.human_symbol.map(human_var.gene_ids), orthologs.mouse_symbol))
    by_symbol = dict(zip(orthologs.human_symbol, orthologs.mouse_symbol))
    target = np.array([by_id.get(i, by_symbol.get(s, '')) for i, s in zip(var_ids, var_names)])
    mapped = target != ''
    frame = pd.DataFrame(sums[:, mapped], columns=target[mapped]).T.groupby(level=0).sum().T.reindex(columns=gene_space, fill_value=0.)
    state_columns = ['PT-S1', 'PT-S2', 'PT-S3', 'aPT', 'dPT', 'cycPT']
    state_counts = state_counts.reindex(columns=state_columns, fill_value=0)
    return {'groups': groups.astype(str), 'nuclei': nuclei, 'totals': totals, 'sums': frame.to_numpy(),
            'state_donors': state_counts.index.to_numpy().astype(str), 'state_counts': state_counts.to_numpy()}


lake = cached_payload('lake_pseudobulk', lake_pseudobulk, root=cache,
                      params={'labels': 'PT-S1,PT-S2,PT-S3', 'region': 'C, cortex of kidney', 'disease': 'all'},
                      inputs={'lake': [lake_file.name, lake_file.stat().st_size], 'genes': list(gene_space),
                              'orthologs': digest(orthologs)}, code=code)
lake_meta = pd.DataFrame([g.split('|') for g in lake['groups']], columns=['donor', 'disease', 'sex', 'segment'])
lake_meta['nuclei'], lake_meta['total'] = lake['nuclei'], lake['totals']
lake_sums = pd.DataFrame(lake['sums'], columns=gene_space)
nuclei_table = lake_meta.pivot_table(index='donor', columns='segment', values='nuclei', aggfunc='sum', fill_value=0)
lake_donors = nuclei_table.index[(nuclei_table.reindex(columns=['S1', 'S2', 'S3'], fill_value=0) >= MIN_NUCLEI).all(axis=1)]
lake_disease = lake_meta.drop_duplicates('donor').set_index('donor').disease
lake_states = pd.DataFrame(lake['state_counts'], index=lake['state_donors'], columns=['PT-S1', 'PT-S2', 'PT-S3', 'aPT', 'dPT', 'cycPT'])
lake_counts = {}
lake_totals = {}
for donor in lake_donors:
    for seg_name in ('S1', 'S2', 'S3'):
        rows = lake_meta.index[lake_meta.donor.eq(donor) & lake_meta.segment.eq(seg_name)]
        lake_counts[(donor, seg_name)] = lake_sums.loc[rows].sum()
        lake_totals[(donor, seg_name)] = float(lake_meta.loc[rows, 'total'].sum())
lake_counts, lake_totals = pd.DataFrame(lake_counts), pd.Series(lake_totals)
print(f'Lake cortex donors with ≥ {MIN_NUCLEI} nuclei in each segment: {len(lake_donors)} '
      f'({lake_disease.loc[lake_donors].value_counts().to_dict()}); mouse snRNA donors: {sn_sex.value_counts().to_dict()}')

In [ ]:
def log2_cpm(counts_series, total):
    return np.log2(counts_series / total * 1e6 + 1)


def donor_contrasts(count_frame, totals, donors, contrast):
    """Per-donor log2 CPM contrast and per-segment levels; early pools S1 + S2 counts."""
    late, early = CONTRASTS[contrast]
    late = 'S3' if late == 'S3c' else late
    rows_contrast, rows_level = {}, {}
    for donor in donors:
        def level(parts):
            pooled = sum(count_frame[(donor, p)] for p in parts)
            return log2_cpm(pooled, sum(totals[(donor, p)] for p in parts))
        late_level = level([late])
        early_level = level(['S1', 'S2'] if early == 'early' else [early])
        rows_contrast[donor] = late_level - early_level
        rows_level[donor] = np.maximum(late_level, early_level)
    return pd.DataFrame(rows_contrast).T, pd.DataFrame(rows_level).T


def reference_from_donors(count_frame, totals, donors, contrast):
    """Reference gradient (mean over donors), expressed-gene mask and a donor-resampling draw function."""
    return donor_reference(*donor_contrasts(count_frame, totals, donors, contrast))


def reference_from_census_human():
    """Census human cortex: S3 minus convoluted PT per donor (only for the S3 contrasts)."""
    donors = census_h_sex.index
    late = {d: log2_cpm(census_h_counts[(d, 'S3')], census_h_totals[(d, 'S3')]) for d in donors}
    early = {d: log2_cpm(census_h_counts[(d, 'early')], census_h_totals[(d, 'early')]) for d in donors}
    per_donor = pd.DataFrame({d: late[d] - early[d] for d in donors}).T
    level = pd.DataFrame({d: np.maximum(late[d], early[d]) for d in donors}).T
    return donor_reference(per_donor, level)


def donor_reference(per_donor, level):
    expressed = level.mean().ge(REFERENCE_LEVEL)

    def draw(rng):
        rows = np.arange(len(per_donor)) if rng is None else rng.integers(0, len(per_donor), len(per_donor))
        return per_donor.iloc[rows].mean()[expressed]

    error_variance = float((per_donor.loc[:, expressed].var(ddof=1) / len(per_donor)).mean())
    return {'gradient': per_donor.mean()[expressed], 'draw': draw, 'error_variance': error_variance,
            'n_donors': len(per_donor), 'per_donor': per_donor}


def reference_from_micro_male(contrast):
    late, early = CONTRASTS[contrast]
    late = 'S3' if late == 'S3c' else late
    parts_early = ['S1', 'S2'] if early == 'early' else [early]
    by_segment = {s: micro_male[[c for c in micro_male.columns if c[0] == s]] for s in ('S1', 'S2', 'S3')}

    def gradient(rng):
        means = {}
        for s, frame in by_segment.items():
            pick = np.arange(frame.shape[1]) if rng is None else rng.integers(0, frame.shape[1], frame.shape[1])
            means[s] = frame.iloc[:, pick].mean(axis=1)
        return means[late] - sum(means[p] for p in parts_early) / len(parts_early), means

    observed, means = gradient(None)
    expressed = pd.concat([means[late], *[means[p] for p in parts_early]], axis=1).max(axis=1).ge(REFERENCE_LEVEL)
    error_variance = float(sum((by_segment[s].var(axis=1, ddof=1) / by_segment[s].shape[1] * (1 if s == late else 1 / len(parts_early) ** 2))
                               for s in {late, *parts_early})[expressed].mean())
    return {'gradient': observed[expressed], 'draw': lambda rng: gradient(rng)[0][expressed],
            'error_variance': error_variance, 'n_donors': int(min(f.shape[1] for f in by_segment.values()))}


def reference_from_micro_sex(sex, contrast):
    late, early = CONTRASTS[contrast]
    late = 'S3' if late == 'S3c' else late
    table = micro_sex[sex]
    early_value = table[['S1', 'S2']].mean(axis=1) if early == 'early' else table[early]
    gradient = table[late] - early_value
    expressed = pd.concat([table[late], early_value], axis=1).max(axis=1).ge(REFERENCE_LEVEL)
    return {'gradient': gradient[expressed], 'draw': lambda rng: gradient[expressed], 'error_variance': np.nan, 'n_donors': 0}



## 5 · Moderated per-species contrasts and same-species reference tests

- **Our contrasts.** Replicates are the two specimens of a species. Expression bins come from
  that species' mean log2 CPM over the segments being contrasted.
- **Reference contrasts.** Replicates are donors: Census mouse snRNA male, and Lake healthy cortex.
- **Microdissection male** (A5 only). Segments come from different mice, so the gene-level mean
  gradient and its size are used.

In [ ]:
from pseudospace import zonation_classes as classes_module
from pseudospace.zonation_classes import moderated_contrast, zonation_calls, confirm, cross_species_class, union_class

PRIMARY_CONTRASTS = ('S2-S1', 'S3c-early')
ALL_CONTRASTS = ('S2-S1', 'S3c-early', 'S3-early')


def species_level(pb, pb_library, specimens, parts):
    return pd.concat([log2_cpm(pb[(n, p)], pb_library[(n, p)]) for n in specimens for p in parts], axis=1).mean(axis=1)


def our_contrast_table(contrast, specimens, genes):
    replicates = ours.loc[genes, [(contrast, n) for n in specimens]].T.reset_index(drop=True)
    return moderated_contrast(replicates, species_level(ours_counts, ours_library, specimens, list(CONTRASTS[contrast])))


def reference_table(count_frame, totals, donors, contrast):
    per_donor, level = donor_contrasts(count_frame, totals, donors, contrast)
    per_donor = per_donor.loc[:, level.mean().ge(REFERENCE_LEVEL) & per_donor.notna().all()]
    return moderated_contrast(per_donor, level.mean())


healthy_lake = [d for d in lake_donors if lake_disease[d] == 'normal']
male_sn = sn_sex.index[sn_sex.eq('male')]
female_sn = sn_sex.index[sn_sex.eq('female')]
tables_ours, tables_ref = {}, {}
for contrast in ALL_CONTRASTS:
    late, early = CONTRASTS[contrast]
    both = gene_space[(count_filter(ours_counts, expected['mouse'], [late, early]) & count_filter(ours_counts, expected['human'], [late, early])).to_numpy()]
    tables_ours[(contrast, 'mouse')] = our_contrast_table(contrast, expected['mouse'], both)
    tables_ours[(contrast, 'human')] = our_contrast_table(contrast, expected['human'], both)
    tables_ref[(contrast, 'mouse')] = reference_table(sn_counts, sn_totals, male_sn, contrast)
    tables_ref[(contrast, 'mouse female')] = reference_table(sn_counts, sn_totals, female_sn, contrast)
    tables_ref[(contrast, 'human')] = reference_table(lake_counts, lake_totals, healthy_lake, contrast)
print({c: len(tables_ours[(c, 'mouse')]) for c in ALL_CONTRASTS}, 'genes measurable in both species')
print({k: len(v) for k, v in tables_ref.items()}, 'genes expressed in each reference')

## 6 · Calls and classes: primary and alternatives

In [ ]:
micro_gradient = {('male', c): reference_from_micro_male(c)['gradient'] for c in ALL_CONTRASTS}
micro_gradient.update({('female', c): reference_from_micro_sex('female', c)['gradient'] for c in ALL_CONTRASTS})


def amplitude_ratio_reference_free(contrast):
    genes = tables_ours[(contrast, 'mouse')].index
    def signal_var(specimens):
        reps = ours.loc[genes, [(contrast, n) for n in specimens]].to_numpy()
        return reps.mean(axis=1).var(ddof=1) - reps.var(axis=1, ddof=1).mean() / reps.shape[1]
    return float(np.sqrt(max(signal_var(expected['human']), 0) / signal_var(expected['mouse'])))


amplitude = {c: amplitude_ratio_reference_free(c) for c in ALL_CONTRASTS}
print('Noise-corrected human ÷ mouse amplitude:', {k: round(v, 3) for k, v in amplitude.items()})

SETTINGS = {
    'primary': dict(effect=(EFFECT, EFFECT), margin=MARGIN, confirm='snRNA'),
    'A1 no confirmation': dict(effect=(EFFECT, EFFECT), margin=MARGIN, confirm=None),
    'A2 no effect floor': dict(effect=(0., 0.), margin=MARGIN, confirm='snRNA'),
    'A3 amplitude-scaled floor': dict(effect=('scaled', EFFECT), margin=MARGIN, confirm='snRNA'),
    'A4 flat margin 0.5': dict(effect=(EFFECT, EFFECT), margin=.5, confirm='snRNA'),
    'A5 mouse confirmed by microdissection': dict(effect=(EFFECT, EFFECT), margin=MARGIN, confirm='micro male'),
    'A6 mouse confirmed by female snRNA': dict(effect=(EFFECT, EFFECT), margin=MARGIN, confirm='snRNA female'),
    'A7 mouse confirmed by female microdissection': dict(effect=(EFFECT, EFFECT), margin=MARGIN, confirm='micro female')}


def species_calls(contrast, species, effect, margin, confirm_by):
    table = tables_ours[(contrast, species)]
    calls, q = zonation_calls(table, alpha=ALPHA, effect=effect, margin=margin)
    if confirm_by is None:
        return calls, q
    if species == 'human' or confirm_by == 'snRNA':
        calls = confirm(calls, tables_ref[(contrast, species)], alpha=CONFIRM_ALPHA)
    elif confirm_by == 'snRNA female':
        calls = confirm(calls, tables_ref[(contrast, 'mouse female')], alpha=CONFIRM_ALPHA)
    else:
        ref = micro_gradient[(confirm_by.split()[1], contrast)].reindex(calls.index)
        ok = ((calls == 'up') & (ref >= EFFECT)) | ((calls == 'down') & (ref <= -EFFECT))
        calls = calls.where(~calls.isin(['up', 'down']) | ok.fillna(False), 'indeterminate')
    return calls, q


def classify(setting):
    spec = SETTINGS[setting]
    per_contrast = {}
    for contrast in ALL_CONTRASTS:
        human_effect = spec['effect'][0] if spec['effect'][0] != 'scaled' else EFFECT * amplitude[contrast]
        mouse_calls, _ = species_calls(contrast, 'mouse', spec['effect'][1], spec['margin'], spec['confirm'])
        human_calls, _ = species_calls(contrast, 'human', human_effect, spec['margin'], spec['confirm'])
        per_contrast[contrast] = pd.Series([cross_species_class(m, h) for m, h in zip(mouse_calls, human_calls)],
                                           index=mouse_calls.index)
        per_contrast[contrast + ' mouse call'], per_contrast[contrast + ' human call'] = mouse_calls, human_calls
    frame = pd.DataFrame(per_contrast)
    frame['class'] = union_class(frame[list(PRIMARY_CONTRASTS)])
    return frame


probe = pd.read_csv(probe_file).drop_duplicates('mouse_symbol').set_index('mouse_symbol')
balanced = set(probe.index[probe.probe_balance.eq('balanced')])
classified = {setting: classify(setting) for setting in SETTINGS}
primary = classified['primary']
primary['probe_balanced'] = primary.index.isin(balanced)
primary.to_csv(tables / 'gene_zonation_classes.csv')
order_classes = ['conserved', 'reversal', 'mouse-only', 'human-only', 'indeterminate', 'neither']
class_sizes = pd.DataFrame({setting: frame.loc[frame.index.isin(balanced), 'class'].value_counts().reindex(order_classes).fillna(0).astype(int)
                            for setting, frame in classified.items()}).T
per_contrast_sizes = pd.DataFrame({c: primary.loc[primary.probe_balanced, c].value_counts().reindex(order_classes).fillna(0).astype(int)
                                   for c in ALL_CONTRASTS}).T
imbalanced_sizes = primary.loc[~primary.probe_balanced, 'class'].value_counts().reindex(order_classes).fillna(0).astype(int)
class_sizes.to_csv(tables / 'class_sizes_by_setting.csv')
per_contrast_sizes.to_csv(tables / 'class_sizes_by_contrast.csv')
display(class_sizes)
display(per_contrast_sizes)
print('Probe-imbalanced genes (primary, combined class):', imbalanced_sizes.to_dict())
retention_rows = {}
for setting, frame in classified.items():
    if setting == 'primary':
        continue
    row = {}
    for klass in ('mouse-only', 'conserved', 'reversal', 'human-only'):
        genes_ = primary.index[primary['class'].eq(klass) & primary.probe_balanced]
        row[klass] = f"{int(frame.loc[genes_, 'class'].eq(klass).sum())} of {len(genes_)}"
    retention_rows[setting] = row
class_retention = pd.DataFrame(retention_rows).T
class_retention.to_csv(tables / 'class_retention_by_setting.csv')
display(class_retention.rename_axis('primary genes keeping their class'))

### Genes expressed in one species only

These genes have ≥ 20 counts in every pseudobulk of one species and < 20 in every pseudobulk of
the other. The expressing species is called with the same rules and its reference; there is no
cross-species class. A probe gap and true absence look the same here.

In [ ]:
one_species_rows = []
for contrast in PRIMARY_CONTRASTS:
    late, early = CONTRASTS[contrast]
    for species, other in (('mouse', 'human'), ('human', 'mouse')):
        present = count_filter(ours_counts, expected[species], [late, early])
        absent = ours_counts[[(n, p) for n in expected[other] for p in (late, early)]].max(axis=1).lt(MIN_COUNTS)
        genes = gene_space[(present & absent).to_numpy()]
        if len(genes) < 20:
            continue
        table = moderated_contrast(ours.loc[genes, [(contrast, n) for n in expected[species]]].T.reset_index(drop=True),
                                   species_level(ours_counts, ours_library, expected[species], [late, early]))
        calls, _ = zonation_calls(table, alpha=ALPHA, effect=EFFECT, margin=MARGIN)
        calls = confirm(calls, tables_ref[(contrast, species)], alpha=CONFIRM_ALPHA)
        one_species_rows.append(pd.DataFrame({'contrast': contrast, 'expressed_in': species, 'gene': genes,
                                              'call': calls.to_numpy(), 'mean': table['mean'].to_numpy()}))
one_species = pd.concat(one_species_rows, ignore_index=True)
one_species.to_csv(tables / 'one_species_expressed_genes.csv', index=False)
display(one_species.groupby(['contrast', 'expressed_in']).call.value_counts().unstack(fill_value=0))

## 7 · Pathway enrichment per class

- Gene statistic: 1 for a class member, 0 otherwise, over all classified genes.
- Matched competitive rank-AUC (strata: mean expression, detection, minimum species detection),
  with a CAMERA VIF.
- The VIF comes from residual correlations after removing each specimen × segment mean, so it
  needs no coordinate.
- Sensitivities: 5-bin strata, and excluding the lowest expression tertile.

In [ ]:
import ast
from pseudospace.pathway_remodeling import matching_strata, matched_pathway_tests, residual_pathway_correlations, correlation_adjusted_rank_tests
from pseudospace.pathways import build_pathway_membership
from pseudospace.pathway_decoys import joint_matched_test, overlap_programs
from statsmodels.stats.multitest import multipletests

universe_genes = primary.index[primary['class'].notna()]
libraries = ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')
membership = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{library}.json').read_text()),
    gene_space, ortholog_map=orthologs, library_name=library, tested=list(universe_genes), min_genes=1)
    for library in libraries], ignore_index=True)
membership['pathway_id'] = membership.library + '::' + membership.pathway
full_membership = membership.copy()
membership = membership[membership.n_tested.between(10, 300)]
class_sets = {row.pathway_id: [g for g in row.genes_present if g in set(universe_genes)] for row in membership.itertuples()}
class_sets = {p: v for p, v in class_sets.items() if 10 <= len(v) <= 300}
pathway_names = pd.Series({p: p.split('::', 1)[1] for p in class_sets}).str.replace(r' R-HSA-\d+$', '', regex=True)
print(f'{len(universe_genes):,} classified genes · {len(class_sets):,} pathways')

column_index = gene_space.get_indexer(universe_genes)
structure_values = lognorm[:, column_index]
mean_expression = pd.Series(np.asarray(structure_values.mean(axis=0)).ravel(), index=universe_genes)
detection = pd.Series(np.asarray((structure_values > 0).mean(axis=0)).ravel(), index=universe_genes)
min_detection = pd.Series(np.minimum(*[np.asarray((structure_values[np.isin(specimen, expected[s])] > 0).mean(axis=0)).ravel()
                                        for s in ('mouse', 'human')]), index=universe_genes)
covariates = pd.DataFrame({'mean_expression': mean_expression, 'detection': detection, 'coverage': min_detection})
strata3, strata5 = matching_strata(covariates, bins=3), matching_strata(covariates, bins=5)


def residual_correlations():
    dense = structure_values.toarray()
    groups = pd.Series(pd.Series(specimen).astype(str) + '|' + pd.Series(segment_region).astype(str))
    for _, idx in groups.groupby(groups).groups.items():
        rows = np.asarray(idx)
        dense[rows] -= dense[rows].mean(axis=0)
    return residual_pathway_correlations(dense, specimen, list(universe_genes), class_sets)


correlations = cached_frame('class_residual_correlations', residual_correlations, root=cache,
                            inputs={'genes': list(universe_genes), 'sets': class_sets, 'structures': len(specimen)}, code=code)
indicator = pd.DataFrame({k: primary.loc[universe_genes, 'class'].eq(k).astype(float) for k in ('conserved', 'reversal', 'mouse-only', 'human-only')})
vif = correlation_adjusted_rank_tests(indicator['conserved'], class_sets, correlations).set_index('pathway_id').variance_inflation
upper = mean_expression.index[mean_expression.ge(mean_expression.quantile(1 / 3))]
upper_sets = {p: [g for g in v if g in set(upper)] for p, v in class_sets.items()}
upper_sets = {p: v for p, v in upper_sets.items() if len(v) >= 10}
enrich = {}
for label, frame_, sets_, strata_ in (('primary', indicator, class_sets, strata3), ('5-bin', indicator, class_sets, strata5),
                                      ('upper expression', indicator.loc[upper], upper_sets, pd.Series(strata3, index=universe_genes).loc[upper].to_numpy())):
    result = cached_frame(f'class_enrichment_{label.replace(" ", "_")}', lambda frame_=frame_, sets_=sets_, strata_=strata_:
                          matched_pathway_tests(frame_, sets_, strata_, n_null=N_NULL, seed=SEED), root=cache,
                          inputs={'indicator': frame_, 'sets': sets_, 'strata': strata_}, code=code)
    enrich[label] = joint_matched_test(result, vif, keys=('statistic',))
enrichment = enrich['primary'].copy()
enrichment = enrichment.merge(enrich['5-bin'][['pathway_id', 'statistic', 'q_joint']].rename(columns={'q_joint': 'q_5bin'}),
                              on=['pathway_id', 'statistic'], how='left')
enrichment = enrichment.merge(enrich['upper expression'][['pathway_id', 'statistic', 'q_joint']].rename(columns={'q_joint': 'q_upper_expression'}),
                              on=['pathway_id', 'statistic'], how='left')
enrichment['name'] = enrichment.pathway_id.map(pathway_names)
enrichment['class_members'] = [int(indicator.loc[class_sets[p], c].sum()) for p, c in zip(enrichment.pathway_id, enrichment.statistic)]
enrichment['enriched'] = enrichment.effect.gt(0) & enrichment.q_joint.le(ENRICH_Q)
enrichment['not_expression_driven'] = enrichment.enriched & enrichment.q_5bin.le(ENRICH_Q) & enrichment.q_upper_expression.le(ENRICH_Q)
programs = []
for klass in indicator.columns:
    ids = enrichment.loc[enrichment.statistic.eq(klass) & enrichment.enriched, 'pathway_id'].tolist()
    members = set(indicator.index[indicator[klass].eq(1)])
    if ids:
        groups_ = overlap_programs(class_sets, ids, contributing=members, cut=.5).assign(statistic=klass)
        programs.append(groups_)
programs = pd.concat(programs, ignore_index=True) if programs else pd.DataFrame(columns=['pathway_id', 'program', 'statistic'])
enrichment = enrichment.merge(programs, on=['pathway_id', 'statistic'], how='left')
enrichment.sort_values(['statistic', 'q_joint']).to_csv(tables / 'class_pathway_enrichment.csv', index=False)
summary_enrichment = enrichment.groupby('statistic').agg(enriched=('enriched', 'sum'), not_expression_driven=('not_expression_driven', 'sum'),
                                                         programs=('program', 'nunique'))
display(summary_enrichment)


def program_summary(klass, top=6):
    block = enrichment[enrichment.statistic.eq(klass) & enrichment.enriched]
    rows = []
    for program, group in block.groupby('program'):
        lead = group.sort_values('q_joint').iloc[0]
        genes_ = pd.Series([g for p in group.pathway_id for g in class_sets[p] if indicator.loc[g, klass] == 1]).value_counts()
        rows.append({'class': klass, 'program': int(program), 'name': lead['name'], 'n_pathways': len(group),
                     'best_q': lead.q_joint, 'robust_to_expression': bool(group.not_expression_driven.any()),
                     'genes': ', '.join(genes_.index[:8])})
    return pd.DataFrame(rows).sort_values('best_q').head(top) if rows else pd.DataFrame()


programs_by_class = pd.concat([program_summary(k) for k in indicator.columns], ignore_index=True)
programs_by_class.to_csv(tables / 'class_programs.csv', index=False)
display(programs_by_class)

### Is class membership tied to expression or detection?

This is descriptive. It shows class shares by expression tertile; the matched test above is the
control.

In [ ]:
tertile = pd.qcut(mean_expression, 3, labels=['low', 'middle', 'high'])
composition = pd.crosstab(tertile, primary.loc[universe_genes, 'class'], normalize='index').reindex(columns=order_classes).round(3)
composition.to_csv(tables / 'class_by_expression_tertile.csv')
display(composition)

## 8 · The 36 robust pathways and the lead genes

In [ ]:
final37 = pd.read_csv(upstream37 / 'tables' / 'table_S2_pathways.csv', index_col=0)
robust_ids = final37.index[final37.robust.astype(bool)]
robust_rows = []
all_members = {row.pathway_id: list(row.genes_present) for row in full_membership.itertuples()}
for p in robust_ids:
    members = [g for g in all_members.get(p, []) if g in set(universe_genes)]
    if not members:
        robust_rows.append({'pathway_id': p, 'name': final37.loc[p, 'name'], 'core': bool(final37.loc[p, 'core']),
                            'program_nb37': final37.loc[p, 'program'], 'classified_members': 0,
                            'dominant_zonated_class': 'not classifiable', 'enriched_classes': '',
                            'note': 'no member measurable in both species'})
        continue
    shares = primary.loc[members, 'class'].value_counts(normalize=True).reindex(order_classes).fillna(0)
    zonated = shares[['conserved', 'reversal', 'mouse-only', 'human-only']]
    enriched_in = enrichment.loc[enrichment.pathway_id.eq(p) & enrichment.enriched, 'statistic'].tolist()
    robust_rows.append({'pathway_id': p, 'name': final37.loc[p, 'name'], 'core': bool(final37.loc[p, 'core']),
                        'program_nb37': final37.loc[p, 'program'], 'classified_members': len(members),
                        **{f'share_{k}': shares[k] for k in order_classes},
                        'dominant_zonated_class': zonated.idxmax() if zonated.max() > 0 else 'none',
                        'enriched_classes': ', '.join(enriched_in),
                        'note': '' if p in class_sets else f'{len(members)} classified members: below the 10-member enrichment floor'})
robust_map = pd.DataFrame(robust_rows).set_index('pathway_id')
robust_map.to_csv(tables / 'robust_pathways_by_class.csv')
display(robust_map[['name', 'classified_members', 'share_conserved', 'share_reversal', 'share_mouse-only', 'share_human-only',
                    'dominant_zonated_class', 'enriched_classes', 'note']].round(2))
display(robust_map.dominant_zonated_class.value_counts())

lead_genes = ['Gatm', 'Acadm', 'Hadh', 'Acaa2', 'Gclc', 'Gclm', 'Gss', 'Hmgcs1', 'Cyp51', 'Hmgcr', 'Slc22a6', 'Slc22a8',
              'Slc13a3', 'Slc6a19', 'Dcxr', 'Pah', 'Igfbp4', 'Ugt3a1', 'Acox2', 'Nt5e', 'Cyp24a1', 'Phgdh', 'Psat1', 'Ugt1a9',
              'Ephx1', 'Rbp4', 'Aox1', 'Inmt', 'Lpl', 'Pank1', 'Cyp2e1', 'Acsm3', 'Crot', 'Nudt19', 'Slc27a2', 'Cyp7b1',
              'Cyp4b1', 'Slc22a7', 'Slc22a13', 'Slc5a8', 'Slc5a2', 'Slc34a1', 'Lrp2', 'Cubn']
lead_rows = {}
for g in lead_genes:
    if g in primary.index:
        row = primary.loc[g]
        lead_rows[g] = {'class': row['class'], **{c: row[c] for c in ALL_CONTRASTS}, 'probe_balanced': g in balanced,
                        'mouse S2−S1': tables_ours[('S2-S1', 'mouse')]['mean'].get(g, np.nan),
                        'human S2−S1': tables_ours[('S2-S1', 'human')]['mean'].get(g, np.nan)}
    else:
        hit = one_species[one_species.gene.eq(g)]
        lead_rows[g] = {'class': f'expressed in {hit.expressed_in.iloc[0]} only ({hit.call.iloc[0]})' if len(hit) else 'not measurable',
                        'probe_balanced': g in balanced}
lead_table = pd.DataFrame(lead_rows).T
lead_table.to_csv(tables / 'lead_gene_classes.csv')
display(lead_table)

### Reading the classes

- **Most genes are not classifiable with two specimens per species.** With the noise-scaled
  rules, most genes are indeterminate. Among genes that are classifiable, mouse-only zonation
  outnumbers human-only under every setting. The gap narrows when the human effect floor is
  scaled to human amplitude (A3).
- **The conserved core** is apical transport: amino-acid, organic-anion and phosphate
  transporters (Slc3a1, Slc7a8, Slc6a18, Slc22a7, Slc22a12, Slc34a3, Slc5a2), plus the renal
  renin–angiotensin genes. Its enrichment survives 5-bin matching and removal of low-expression
  genes.
- **Mouse-only programs** are sterol synthesis, peroxisomal very-long-chain β-oxidation, tyrosine
  catabolism and pyrimidine catabolism. Lead genes in this class: Gatm, Gclc/Gclm, Cyp51,
  Slc22a6, Slc13a3, Acox2, Slc27a2.
- **Human-only and reversal classes** are small, and their pathway enrichments are mostly not
  robust to expression. The human side is better described gene by gene: reversals such as
  Dcxr and Ugt3a1, and genes expressed only in human (Rbp4, Aox1, Ugt1a9).
- **The 36 robust pathways** split mainly between mouse-only-dominant and conserved-dominant
  member compositions (counts in the table above). The shares are small because most members
  are indeterminate. Pathways with fewer than 10 classified members still appear in the table,
  with a note; a pathway with no classified member is listed as not classifiable.

## 9 · Representative genes, by the pre-set rule

**Amendment (made after the first run).** The pre-set rule first picked Egf, which notebook 37's
literature check flags as neighbouring-segment RNA. Genes on that flagged list (ambient,
neighbouring-segment or mapping artefacts) are therefore excluded from the representatives.
Class calls and counts are unchanged.

In [ ]:
FLAGGED_GENES = ['Alb', 'Ttr', 'Apoa1', 'Apoa2', 'Apoh', 'Fgb', 'Fga', 'Fgg', 'Egf', 'Muc1', 'Klk1', 'Slc26a7', 'Car12',
                 'Akr1c18', 'Ugt1a1', 'Ugt1a9']
eligible_rep = universe_genes[universe_genes.isin(balanced) & ~universe_genes.isin(FLAGGED_GENES)
                              & mean_expression.reindex(universe_genes).ge(mean_expression.median()).to_numpy()]


def representatives(klass, n=3):
    genes_ = eligible_rep[primary.loc[eligible_rep, 'class'].eq(klass).to_numpy()]
    if not len(genes_):
        return []
    scores = []
    for g in genes_:
        contrast = next(c for c in PRIMARY_CONTRASTS if primary.loc[g, c] == klass)
        tm, th = abs(tables_ours[(contrast, 'mouse')].t[g]), abs(tables_ours[(contrast, 'human')].t[g])
        scores.append({'conserved': min(tm, th), 'reversal': min(tm, th), 'mouse-only': tm, 'human-only': th}[klass])
    return list(pd.Series(scores, index=genes_).sort_values(ascending=False).index[:n])


representative = {k: representatives(k) for k in ('conserved', 'reversal', 'mouse-only', 'human-only')}
print(representative)

## 10 · Figure: the R3 main figure

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 8.5,
                     'axes.titlesize': 9.5, 'axes.titleweight': 'bold', 'pdf.fonttype': 42})
HUMAN, MOUSE, INK, NULL = '#D55E00', '#0072B2', '#333333', '#9A9A9A'
CLASS_COLORS = {'conserved': '#009E73', 'reversal': '#CC79A7', 'mouse-only': MOUSE, 'human-only': HUMAN,
                'indeterminate': '#DDDDDD', 'neither': '#BBBBBB'}


def save(fig, name):
    for suffix in ('pdf', 'png'):
        fig.savefig(figures / f'{name}.{suffix}', bbox_inches='tight', dpi=220)


fig = plt.figure(figsize=(15, 13.5))
grid_spec = fig.add_gridspec(3, 4, height_ratios=[1.15, 1, 1.7], hspace=.5, wspace=.45)
ax = fig.add_subplot(grid_spec[0, :2])
balanced_s21 = tables_ours[('S2-S1', 'mouse')].index[tables_ours[('S2-S1', 'mouse')].index.isin(balanced)]
tm, th = tables_ours[('S2-S1', 'mouse')].loc[balanced_s21], tables_ours[('S2-S1', 'human')].loc[balanced_s21]
s21_class = primary.loc[balanced_s21, 'S2-S1']
for klass in ['neither', 'indeterminate', 'mouse-only', 'human-only', 'conserved', 'reversal']:
    mask = s21_class.eq(klass)
    ax.scatter(tm['mean'][mask], th['mean'][mask], s=4 if klass in ('neither', 'indeterminate') else 8,
               color=CLASS_COLORS[klass], alpha=.5 if klass in ('neither', 'indeterminate') else .85, rasterized=True,
               label=f'{klass} ({int(mask.sum()):,})')
ax.axhline(0, color=NULL, lw=.6)
ax.axvline(0, color=NULL, lw=.6)
ax.set(xlabel='Mouse S2 − S1 (log2, mean of 2 mice)', ylabel='Human S2 − S1 (log2, mean of 2 sections)')
ax.legend(frameon=False, fontsize=7, markerscale=2, loc='upper left')
ax.set_title(f'a · S1 → S2 class per gene ({len(balanced_s21):,} probe-balanced genes measurable in both species)', loc='left')
ax = fig.add_subplot(grid_spec[0, 2:])
plot_sizes = class_sizes[['conserved', 'reversal', 'mouse-only', 'human-only']]
left = np.zeros(len(plot_sizes))
for klass in plot_sizes.columns:
    ax.barh(np.arange(len(plot_sizes))[::-1], plot_sizes[klass], left=left, color=CLASS_COLORS[klass], label=klass, height=.6)
    left += plot_sizes[klass].to_numpy()
for yi, (name, row) in zip(np.arange(len(plot_sizes))[::-1], plot_sizes.iterrows()):
    ax.text(left[list(plot_sizes.index).index(name)] + 5, yi, ' / '.join(str(int(v)) for v in row), va='center', fontsize=7)
ax.set_yticks(np.arange(len(plot_sizes))[::-1], plot_sizes.index, fontsize=7.5)
ax.set_xlabel('Probe-balanced genes (combined S2 − S1 and S3c − early class)')
ax.legend(frameon=False, fontsize=7, ncol=4, loc='lower right', bbox_to_anchor=(1, 1.0))
ax.set_title('b · Class sizes under the primary rule and alternatives', loc='left', pad=18)
reps = [(k, g) for k in ('conserved', 'reversal', 'mouse-only', 'human-only') for g in representative[k][:2]]
cards = grid_spec[1, :].subgridspec(1, max(len(reps), 1), wspace=.45)
for k, (klass, gene) in enumerate(reps):
    ax = fig.add_subplot(cards[0, k])
    for name in names:
        values = [log2_cpm(ours_counts.loc[gene, (name, p)], ours_library[(name, p)]) for p in ('S1', 'S2', 'S3c')]
        ax.plot([1, 2, 3], values, marker='o', ms=3, lw=1.3, color=HUMAN if name in expected['human'] else MOUSE,
                ls='-' if name in (names[0], names[2]) else '--')
    ax.set_xticks([1, 2, 3], ['S1', 'S2', 'S3c'], fontsize=7)
    ax.tick_params(labelsize=7)
    ax.set_title(f'{gene}', fontsize=8.5, loc='left', style='italic', color=CLASS_COLORS[klass])
    ax.text(0, 1.13, klass, transform=ax.transAxes, fontsize=7, color=CLASS_COLORS[klass], fontweight='bold')
    if k == 0:
        ax.set_ylabel('log2 CPM (each specimen)', fontsize=7.5)
fig.text(.12, .665, 'c · Representative genes (rule-selected): orange human sections, blue mice; S3c = cortical-like mouse S3',
         fontsize=8.5, fontweight='bold')
ax = fig.add_subplot(grid_spec[2, :2])
top = programs_by_class.groupby('class').head(4)
yy = np.arange(len(top))[::-1]
ax.barh(yy, -np.log10(top.best_q.clip(lower=1e-10)), color=[CLASS_COLORS[c] for c in top['class']], height=.6)
ax.set_yticks(yy, [f'{n[:42]}' + (' ✓' if r else '') for n, r in zip(top.name, top.robust_to_expression)], fontsize=7)
ax.axvline(-np.log10(ENRICH_Q), color=NULL, ls='--', lw=.8)
ax.set_xlabel('−log10 q (joint matched test); ✓ also with 5-bin matching and without low-expression genes')
ax.set_title('d · Programs enriched in each class (gene-overlap collapsed)', loc='left')
ax = fig.add_subplot(grid_spec[2, 2:])
share_columns = ['share_conserved', 'share_reversal', 'share_mouse-only', 'share_human-only', 'share_indeterminate', 'share_neither']
rm = robust_map.fillna({c: 0 for c in robust_map.columns if c.startswith('share_')}).sort_values('share_mouse-only')
left = np.zeros(len(rm))
for column in share_columns:
    klass = column.replace('share_', '')
    ax.barh(np.arange(len(rm)), rm[column], left=left, color=CLASS_COLORS[klass], height=.75, label=klass)
    left += rm[column].to_numpy()
ax.set_yticks(np.arange(len(rm)), [('★ ' if c else '') + n[:40] for n, c in zip(rm.name, rm.core)], fontsize=6.5)
ax.set_xlabel('Share of classified members')
ax.legend(frameon=False, fontsize=6.5, ncol=3, loc='lower right', bbox_to_anchor=(1, 1.0))
ax.set_title('e · The 36 robust pathways by member class (★ core)', loc='left', pad=22)
save(fig, 'fig_R3_zonation_classes')
plt.show()
plt.close(fig)

## 11 · Table for R3 and run record

In [ ]:
r3_table = pd.DataFrame({
    'genes (primary, probe-balanced)': class_sizes.loc['primary'],
    'range across alternatives': [f'{int(class_sizes[c].min())}–{int(class_sizes[c].max())}' for c in class_sizes.columns],
    'S2 − S1 only': per_contrast_sizes.loc['S2-S1'], 'S3c − early only': per_contrast_sizes.loc['S3c-early'],
    'enriched pathways': summary_enrichment.enriched.reindex(order_classes),
    'programs': summary_enrichment.programs.reindex(order_classes),
    'top programs': [', '.join(programs_by_class.loc[programs_by_class['class'].eq(c), 'name'].head(3)) for c in order_classes],
    'representative genes': [', '.join(representative.get(c, [])) for c in order_classes]})
r3_table.to_csv(tables / 'table_R3_zonation_classes.csv')
display(r3_table)
manifest = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED, 'amplitude_ratio_reference_free': amplitude,
            'class_sizes': class_sizes.to_dict(orient='index'), 'per_contrast_sizes': per_contrast_sizes.to_dict(orient='index'),
            'probe_imbalanced': imbalanced_sizes.to_dict(), 'enrichment': summary_enrichment.to_dict(orient='index'),
            'robust_dominant_class': robust_map.dominant_zonated_class.value_counts().to_dict(),
            'representatives': representative,
            'caveat': 'Two mice and two cortex sections from one human donor; segment labels define every contrast; '
                      'human confirmation uses 7 Lake donors, mouse 12 Census donors.'}
(output / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print(json.dumps({k: manifest[k] for k in ('class_sizes', 'robust_dominant_class')}, indent=2))